# Generated-native LoRA session

This notebook trains adapters through CKE's existing generated forward/backward/AdamW backend. Python declares the plan and submits batches. It does not calculate gradients or update weights.

**Bounded capability:** existing two-layer dense qwen3-template FP32 BUMP artifacts with explicit `rms_eps=1e-5`; fixed token count (2–64) per compiled library. This is not arbitrary pretrained Qwen import, BF16 decoder training, or QLoRA. Token IDs must match the base tokenizer. No replacement synthetic base is created here.

Before starting, build `build/libckernel_engine.so` and set `CKE_REPO`, `CKE_LORA_BASE`, `CKE_LORA_RUN`, `CKE_LORA_RECORDS`, and `CKE_LORA_TOKENS`. The run directory must not exist. Records are JSON lines containing `tokens` and `targets` arrays of the declared length. Set `CKE_LORA_STEPS` optionally (default 40).

In [ ]:
import json, os, sys
from pathlib import Path
repo = Path(os.environ['CKE_REPO']).resolve()
sys.path.insert(0, str(repo / 'version/v7'))
import ckernel_engine as cke
cke.v8.lora_capabilities()


In [ ]:
base = Path(os.environ['CKE_LORA_BASE'])
run_dir = Path(os.environ['CKE_LORA_RUN'])
records = [json.loads(line) for line in Path(os.environ['CKE_LORA_RECORDS']).read_text().splitlines() if line.strip()]
assert records, 'Provide training records; empty data is not training.'
tokens = int(os.environ['CKE_LORA_TOKENS'])
steps = int(os.environ.get('CKE_LORA_STEPS', '40'))
targets = cke.v8.lora_targets(base, tokens=tokens)
print(targets)
plan = cke.v8.prepare_lora(base, tokens=tokens, adapters=[
    cke.v8.LoRAConfig('layer.0.wq', rank=2, alpha=4, name='query', seed=42),
    cke.v8.LoRAConfig('layer.1.wv', rank=3, alpha=6, name='value', seed=43),
])
plan.describe()


Compilation resolves the declared adapters, derives backward operations, plans memory and builds a portable native library. Initialization is prepared before compilation. Every base parameter is frozen; required activation gradients still flow through it. Calling `train_step` executes the compiled forward, loss, backward and AdamW step together.

In [ ]:
with plan.compile(run_dir, learning_rate=0.003, max_steps=steps,
                  dataset_id=json.dumps(records, sort_keys=True)) as session:
    losses = []
    for step in range(steps):
        row = records[step % len(records)]
        losses.append(session.train_step(row['tokens'], row['targets'])['loss'])
    session.save_checkpoint(run_dir / 'checkpoint')
    session.export_adapters(run_dir / 'adapters')
    result = session.evaluate(records[0]['tokens'], records[0]['targets'])
print({'training_loss_first': losses[0], 'training_loss_last': losses[-1],
       'native_updates': result['step'], 'held_out_quality': 'NOT_TESTED'})


Adapter-only reload starts with fresh optimizer state. Use checkpoint loading when continuing a training trajectory. Evaluation computes the current batch loss without updating parameters or moments. Training loss reduction alone is not held-out quality.

In [ ]:
with cke.v8.LoRASession.open(run_dir) as inference:
    inference.load_adapters(run_dir / 'adapters')
    replay = inference.evaluate(records[0]['tokens'], records[0]['targets'])
    assert replay['step'] == 0
    assert (replay['logits'] == result['logits']).all()
with cke.v8.LoRASession.open(run_dir) as resumed:
    resumed.load_checkpoint(run_dir / 'checkpoint')
    assert resumed.evaluate(records[0]['tokens'], records[0]['targets'])['step'] == steps
